In [0]:
# =============================================================================
# NOTEBOOK  : silver_supplier_master
# PURPOSE   : bronze.supplier_master → silver.supplier_master
# LAYER     : Silver (SCD Type 2 — full version history)
# FREQUENCY : Weekly + incremental
# PATTERN   : spark.read + audit watermark (production pattern)
#
# TRANSFORM:
#   - performance_rating   : Double → Decimal(4,2)
#   - on_time_delivery_pct : Double → Decimal(5,2)
#   - scd_hash             : MD5 of all tracked business columns
#
# SCD TYPE 2 LOGIC (single atomic MERGE):
#   Tracked cols : supplier_name, category, performance_rating,
#                  on_time_delivery_pct
#   NEW → INSERT, CHANGED → close old + insert new, SAME → IGNORE
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_SUPPLIER_MASTER_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    md5, concat_ws, row_number
)
from pyspark.sql.types import DecimalType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_supplier_master"]
TARGET_TABLE = cfg["tables"]["silver_supplier_master"]
PIPELINE     = "silver_supplier_master"
 
SCD_TRACKED_COLS = [
    "supplier_name", "category",
    "performance_rating", "on_time_delivery_pct"
]

In [0]:
# ── Read + SCD2 + Write ───────────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("ingested_at") > lit(last_run_time))
        )
    else:
        bronze_df = spark.read.table(SOURCE_TABLE)
 
    rows_read = bronze_df.count()
    print(f"[READ] {rows_read} new bronze rows since last run")
 
    if rows_read == 0:
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0,
                  extra={"versions_opened": "0", "versions_closed": "0",
                         "records_ignored": "0"})
        raise SystemExit(0)
 
    # ── TRANSFORM ─────────────────────────────────────────────────────────────
    df = (
        bronze_df
        .withColumn("performance_rating",
            col("performance_rating").cast(DecimalType(4, 2)))
        .withColumn("on_time_delivery_pct",
            col("on_time_delivery_pct").cast(DecimalType(5, 2)))
        .withColumn("processed_at",    current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
        .withColumn("source_system",   lit("supplier_master_csv"))
    )
 
    window_dedup = (
        Window.partitionBy("supplier_id").orderBy(col("ingested_at").desc())
    )
    df = (
        df
        .withColumn("_row_num", row_number().over(window_dedup))
        .filter(col("_row_num") == 1)
        .drop("_row_num")
    )
 
    df = df.withColumn(
        "scd_hash",
        md5(concat_ws("|", *[col(c).cast("string") for c in SCD_TRACKED_COLS]))
    )
 
    current_silver = (
        spark.read.table(TARGET_TABLE)
        .filter(col("is_current") == True)
        .select("supplier_id", "scd_hash")
        .withColumnRenamed("scd_hash", "existing_hash")
    )
 
    df_classified = df.join(current_silver, "supplier_id", "left")
 
    df_to_process = df_classified.filter(
        col("existing_hash").isNull() |
        (col("existing_hash") != col("scd_hash"))
    ).drop("existing_hash")
 
    rows_to_process = df_to_process.count()
    records_ignored = rows_read - rows_to_process
    print(f"[SCD2] to_process={rows_to_process} | ignored(same)={records_ignored}")
 
    if rows_to_process == 0:
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=rows_read, rows_written=0,
                  extra={"versions_opened": "0", "versions_closed": "0",
                         "records_ignored": str(records_ignored)})
        raise SystemExit(0)
 
    now = current_timestamp()
 
    changed_ids = (
        df_classified
        .filter(col("existing_hash").isNotNull() &
                (col("existing_hash") != col("scd_hash")))
        .select("supplier_id")
    )
 
    close_rows = (
        df_to_process.join(changed_ids, "supplier_id", "inner")
        .withColumn("_action",    lit("close"))
        .withColumn("valid_from", lit(None).cast(TimestampType()))
        .withColumn("valid_to",   now)
        .withColumn("is_current", lit(False))
        .select([f.name for f in SILVER_SUPPLIER_MASTER_SCHEMA.fields] + ["_action"])
    )
 
    insert_rows = (
        df_to_process
        .withColumn("_action",    lit("insert"))
        .withColumn("valid_from", now)
        .withColumn("valid_to",   lit(None).cast(TimestampType()))
        .withColumn("is_current", lit(True))
        .select([f.name for f in SILVER_SUPPLIER_MASTER_SCHEMA.fields] + ["_action"])
    )
 
    merge_source = close_rows.union(insert_rows)
 
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(
            merge_source.alias("s"),
            """
            t.supplier_id = s.supplier_id AND
            t.is_current  = true          AND
            s._action     = 'close'
            """
        )
        .whenMatchedUpdate(set={
            "is_current": "false",
            "valid_to":   "current_timestamp()"
        })
        .whenNotMatchedInsert(
            condition="s._action = 'insert'",
            values={f.name: f"s.{f.name}"
                    for f in SILVER_SUPPLIER_MASTER_SCHEMA.fields}
        )
        .execute()
    )
 
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics").collect()[0][0]
    )
    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))
 
    print(f"[DONE] versions_opened={rows_inserted} | versions_closed={rows_updated} "
          f"| ignored={records_ignored}")
 
    end_audit(
        spark, PROJECT_ROOT, run_id, "SUCCESS",
        rows_read=rows_read, rows_written=rows_inserted,
        extra={
            "versions_opened": str(rows_inserted),
            "versions_closed": str(rows_updated),
            "records_ignored": str(records_ignored)
        }
    )
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)
 
silver_df = spark.read.table(TARGET_TABLE)
print("Total versions    :", silver_df.count())
print("Current active    :", silver_df.filter(col("is_current") == True).count())
print("Null supplier_ids :", silver_df.filter(col("supplier_id").isNull()).count())
 
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics").show(truncate=False)
